In [ ]:
import SAIfunc_20260217 as sai
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pandas as pd

In [ ]:
def plotA(sim):
    v,e = sim.all()
    barmx = 1 + np.arange(sim.len) # sequence of bar centres, for each frame

    type0nos = sim.nV - (type1nos := np.count_nonzero(v, axis=1))

    vdim1, vdim2 = v[:, None,:], v[:, :,None]
    edges11 = np.count_nonzero(np.where((vdim1)&(vdim2), e, 0), axis=(1,2)) / 2
    edges10 = np.count_nonzero(np.where((vdim1)^(vdim2), e, 0), axis=(1,2)) / 2
    edges00 = np.count_nonzero(np.where((~vdim1)&(~vdim2), e, 0), axis=(1,2)) / 2

    # ---------------------
    f = plt.figure(figsize=(16,7), dpi=200, layout='constrained')
    f.suptitle(sim.timestamp)
    axs = f.subplots(nrows=2, ncols=1)

    # ---------------------
    axs[0].set_title('vertex count breakup')
    axs[0].set(xlim=(0.5, sim.len + 0.5), ylim=(0, sim.nV))
    axs[0].set(xlabel='frame index', ylabel='edge count')

    axs[0].bar(barmx, height=type0nos, bottom=type1nos, width=1, color=[(1,.1,.1, 0.7)], label='type0')
    axs[0].bar(barmx, height=type1nos, bottom=0, width=1, color=[(.1,.1,1, 0.7)], label='type1')

    axs[0].grid(which='both', axis='x')
    axs[0].legend()

    # ---------------------

    axs[1].set_title('edge count breakup')
    axs[1].set(xlim=(0.5, sim.len + 0.5))
    axs[1].set(xlabel='frame index', ylabel='edge count')

    axs[1].bar(barmx, height=edges00, bottom=edges11+edges10, width=1, color=[(1,.2,.1, 0.7)], label='0-0')
    axs[1].bar(barmx, height=edges10, bottom=edges11, width=1, color=[(.8,.2,.8, 0.7)], label='0-1')
    axs[1].bar(barmx, height=edges11, bottom=0, width=1, color=[(.1,.4,1, 0.7)], label='1-1')

    axs[1].legend()

    # ---------------------
    f.tight_layout()
    f.savefig(f'figs/plotA-{sim.timestamp}.svg')
    plt.close(f)

In [ ]:
spec = {
    'nV': 40,
    'p0': 0.5,
    'pE': 1/10,
    'a0': 4,
    'a1': 1
}

activesim = sai.Sim(**spec, nT_ini=50, nT_ext=50)

i = 1
while i < 500:
    activesim.next()
    i += 1
activesim.save()
plotA(activesim)

In [ ]:
extend_by = 300
for npz in Path('').glob('data/SAIsim-*.npz'):
    loadedsim = sai.Sim.fromfile(npz, extend_by)
    
    # i=0
    # while i < extend_by:
    #     loadedsim.next()
    #     i += 1
    # loadedsim.save(overwrite=True)
    plotA(loadedsim)
    

In [ ]:
a_ = 2.0 ** np.arange(-3, 4)
print(a_)

for a0 in a_:
    for a1 in a_:
        print(f'<a0:{a0}, a1:{a1}>')
        activesim = sai.Sim(
            nV=40, p0=.5, pE=.1, a0=a0, a1=a1, 
            nT_ini=200, nT_ext=100
        )

        i = 1
        while i < 500:
            activesim.next()
            i += 1
        activesim.save()
        plotA(activesim)